In [1]:
import pandas as pd
import spacy
# from date_spacy import find_dates
import re
from collections import defaultdict

!spacy download en_core_web_sm

In [2]:
# Load the spaCy model for Named Entity Recognition (NER)
nlp = spacy.load("en_core_web_trf")

ruler = nlp.add_pipe("entity_ruler", config={"overwrite_ents": True}, before="ner")

# Add Social Security Number
ssn_pattern_regex = {
    "label": "SSN",
    "pattern": [
        {"TEXT": {"REGEX": "\\d{3}"}},
        {"TEXT": "-"},
        {"TEXT": {"REGEX": "\\d{2}"}},
        {"TEXT": "-"},
        {"TEXT": {"REGEX": "\\d{4}"}}
    ]
}
# Add gender/sex pattern
gender_pattern_regex = {
    "label": "GENDER",
    "pattern": [
        {"LOWER": {"REGEX": "\\b(gender|sex|male|female|man|woman|boy|girl|he|she|him|her)\\b"}}
    ]
}

email_pattern_regex = {
    "label": "EMAIL",
    "pattern": [
        {"TEXT": {"REGEX": "[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"}}
    ]
}

ruler.add_patterns([ssn_pattern_regex, gender_pattern_regex, email_pattern_regex])


<>:28: SyntaxWarning: invalid escape sequence '\.'
<>:28: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_23645/2875841989.py:28: SyntaxWarning: invalid escape sequence '\.'
  {"TEXT": {"REGEX": "[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"}}


In [3]:
def regex_name_fallback(text, redacted_text):
    # Match words that look like names (e.g., lowercase words in a sentence)
    name_like_words = re.findall(r'\b[a-z][a-z]+\b', text)
    for word in name_like_words:
        if word in text and word not in redacted_text:
            redacted_text = re.sub(rf'\b{word}\b', '[]', redacted_text)
    return redacted_text

In [4]:
# Sample dataframe
data = {'comments': [
    "My name is Sarah Johnson and I was born on 1995-04-12 in Chicago, Illinois.",
    "The patient's social security number is 456-78-9012 and their gender is male.",
    "We visited the Eiffel Tower in Paris last summer. My phone number is (555) 123-4567.",
    "John works at Microsoft in Redmond, WA. His email is john.doe@microsoft.com.",
    "I identify as a female. My address is 123 Main St, Springfield, IL 62704.",
    "The meeting was scheduled for 2023-11-15 at 2:00 PM CST. Attendees included Alice Smith and Bob Jones.",
    "My passport number is A12345678 and I'm a citizen of Canada. My salary is $75,000 annually.",
    "We're looking for volunteers in the Seattle area. Please contact us at volunteers@redcross.org.",
    "The event will take place at the Los Angeles Convention Center on July 4th, 2024.",
    "My favorite book is 'The Great Gatsby' and I was born in 1988. My gender is non-binary."
]}


In [5]:
df = pd.DataFrame(data)
df

,comments
0,My name is Sarah Johnson and I was born on 199...
1,The patient's social security number is 456-78...
2,We visited the Eiffel Tower in Paris last summ...
3,"John works at Microsoft in Redmond, WA. His em..."
4,I identify as a female. My address is 123 Main...
5,The meeting was scheduled for 2023-11-15 at 2:...
6,My passport number is A12345678 and I'm a citi...
7,We're looking for volunteers in the Seattle ar...
8,The event will take place at the Los Angeles C...
9,My favorite book is 'The Great Gatsby' and I w...


In [6]:
# Define a function to redact PII using spaCy NER
def redact_pii(text):
    doc = nlp(text)
    redacted_text = text
    for ent in doc.ents:
        if ent.label_ in ["SSN", 
                          "GENDER", 
                          "PERSON", 
                          "NORP", 
                          "FAC", 
                          "ORG", 
                          "GPE", 
                          "LOC", 
                          "PRODUCT", 
                          "EVENT", 
                          "WORK_OF_ART", 
                          "LAW", 
                          "LANGUAGE", 
                          "DATE", 
                          "TIME", 
                          # "PERCENT", 
                          "MONEY", 
                          # "QUANTITY", 
                          # "ORDINAL", 
                          # "CARDINAL"
                          ]:
            redacted_text = redacted_text.replace(ent.text, "[]")
        else:
            redacted_text = redacted_text.replace(ent.text, "[]")
    return redacted_text


In [7]:
# Apply the function to the comments column
for comment in df["comments"].to_list():
    # print(comment)
    print(redact_pii(comment))


My name is [] and I was born on [] in [], [].
The patient's social security number is [] and their [] is [].
We visited [] in [] []. My phone number is [].
[] works at [] in [], []. His email is [].
I identify as a []. My address is 123 Main St, [], [] [].
The meeting was scheduled for [] at []. Attendees included [] and [].
My passport number is A12345678 and I'm a citizen of []. My salary is $[] [].
We're looking for volunteers in the [] area. Please contact us at [].
The event will take place at [] on [].
My favorite book is '[]' and I was born in []. My [] is non-binary.


In [10]:
d = defaultdict(list)
def show_label(text):
    doc = nlp(text)
    for ent in doc.ents:
        d["text"].append(ent.text)
        d["label"].append(ent.label_)
        return ent.text, ent.label_

In [11]:

for comment in df["comments"].to_list():
    # print(comment)
    text, label = show_label(comment)
    

In [12]:
pd.DataFrame(d)

,text,label
0,Sarah Johnson,PERSON
1,456-78-9012,SSN
2,the Eiffel Tower,FAC
3,John,PERSON
4,female,GENDER
5,2023-11-15,DATE
6,Canada,GPE
7,Seattle,GPE
8,the Los Angeles Convention Center,FAC
9,The Great Gatsby,WORK_OF_ART


In [16]:
doc = nlp("prabakharan")
for ent in doc.ents:
    print(ent.text, ent.label_)

prabakharan PERSON
